# Fine-tuned Gemma 4 E2B → LiteRT-LM

Bu notebook, **fine-tune edilmiş** `SalihHub/blind-assist-gemma4-merged-v2` modelini LiteRT-LM formatına çevirir ve resmi modelin **vision encoder**'ını ekleyerek multimodal çalışır hale getirir.

**Akış:**
1. Setup (torch nightly + litert-torch + transformers)
2. Fine-tune modeli `model.litertlm` olarak export et (text-only)
3. Text inference testi
4. Sade modeli HF'ye yükle (ara aşama)
5. Resmi modelin vision sections'ını merge et
6. Final merged modeli HF'ye yükle (vision destekli)

**Çıktı repo:** `SalihHub/blind-assist-gemma4-litert`

## 1. Setup

In [ ]:
import subprocess, sys

def pip(args, desc=''):
    if desc:
        print(f'\n⏳ {desc}...')
    cmd = [sys.executable, '-m', 'pip'] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"pip basarisiz: {' '.join(args[:3])}")
    print('OK')

# litert-torch-nightly torch >= 2.11 gerektiriyor → nightly indeksinden
CUDA_VER = 'cu128'
NIGHTLY_INDEX = f'https://download.pytorch.org/whl/nightly/{CUDA_VER}'

pip(['uninstall', '-y', 'torch', 'torchvision', 'torchaudio', 'torchao',
     'litert-torch', 'litert-torch-nightly'],
    'Eski paketler kaldiriliyor')

pip(['install', 'torch', 'torchvision', 'torchaudio',
     '--index-url', NIGHTLY_INDEX, '--pre', '--quiet'],
    f'torch nightly kuruluyor ({CUDA_VER})')

pip(['install', 'torchao', '--index-url', NIGHTLY_INDEX, '--pre', '--quiet'],
    'torchao nightly kuruluyor')

pip(['install', 'litert-torch-nightly', '--quiet'], 'litert-torch-nightly kuruluyor')
pip(['install', 'litert-lm', '--quiet'], 'litert-lm kuruluyor')
pip(['install', 'protobuf>=6.31.1', '--quiet'], 'protobuf yükseltiliyor')
pip(['install', 'huggingface-hub', 'transformers>=4.50.0', 'accelerate', '--quiet'],
    'HuggingFace araclari kuruluyor')

print('\n' + '=' * 60)
print('KURULUM TAMAM')
print('>>> Runtime -> Restart session (Ctrl+M .) yapip devam et <<<')
print('=' * 60)

#### ---------------- Kernel Restart ----------------

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
    'git+https://github.com/huggingface/transformers.git',
    '--quiet'], check=True)
print('transformers source kuruldu')

In [ ]:
import sys
import google.protobuf
import transformers
import torch
import torchao

print(f'torch         : {torch.__version__}')
print(f'torchao       : {torchao.__version__}')
print(f'protobuf      : {google.protobuf.__version__}')
print(f'transformers  : {transformers.__version__}')

# torchao alt modüllerini önceden yükle (litert_torch import hatalarını önler)
import torchao.quantization
import torchao.quantization.pt2e
import torchao.quantization.pt2e.quantize_pt2e
import torchao.quantization.pt2e.quantizer
print('torchao alt modulleri yuklendi')

try:
    import litert_torch
    print(f'litert_torch {litert_torch.__version__} hazir')
except Exception as e:
    print(f'HATA: {e}')
    import traceback; traceback.print_exc()

## 2. Export Ayarları

In [ ]:
import os
from getpass import getpass

# ============================================================
# ⚙️  FINE-TUNE MODEL AYARLARI
# ============================================================

# Fine-tune edilmiş model (LoRA base'e merge edilmiş olmalı)
MODEL_ID = "SalihHub/blind-assist-gemma4-merged-v2"

# Hedef HF repo (LiteRT-LM çıktısı buraya yüklenecek)
TARGET_REPO = "SalihHub/blind-assist-gemma4-litert"

# Çıktı dizini
OUTPUT_DIR = "/tmp/blind_assist_litert_export"

# Quantization (Google'ın resmi modelinde kullandığı)
QUANTIZE = "dynamic_wi4_afp32"

# Mobil için dengeli ayarlar
PREFILL_SEQ_LEN = 512
CACHE_LENGTH    = 2048
EXTERNALIZE_EMBEDDER = True  # E2B/E4B PLE mimarisi için zorunlu

# Chat template — <pad> sorununu önler
TEMPLATE_REPO = "litert-community/gemma-4-E2B-it-litert-lm"

# ============================================================

# HF Token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN Colab Secrets'tan alındı")
except Exception:
    HF_TOKEN = getpass("HuggingFace Token gir (gizli kalır): ")

os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n📋 Export Ayarları:")
print(f"  Fine-tune model    : {MODEL_ID}")
print(f"  Hedef repo         : {TARGET_REPO}")
print(f"  Çıktı              : {OUTPUT_DIR}")
print(f"  Quantize           : {QUANTIZE}")
print(f"  Prefill seq len    : {PREFILL_SEQ_LEN}")
print(f"  KV Cache length    : {CACHE_LENGTH}")
print(f"  Externalize embed  : {EXTERNALIZE_EMBEDDER}")
print(f"  Chat template      : {TEMPLATE_REPO}")

In [ ]:
from huggingface_hub import hf_hub_download

# Resmi chat template'i indir (preview)
path = hf_hub_download(
    repo_id=TEMPLATE_REPO,
    filename="chat_template.jinja"
)
with open(path) as f:
    template = f.read()
print(template[:500])

## 3. Export — Fine-tune modeli LiteRT-LM'e dönüştür

⏱️ **~30 dakika** sürer (E2B). Colab GPU runtime önerilir.

In [ ]:
import subprocess, time, os

cmd_parts = [
    "litert-torch export_hf",
    MODEL_ID,
    OUTPUT_DIR,
    "--task=text_generation",
]

if EXTERNALIZE_EMBEDDER:
    cmd_parts.append("--externalize_embedder=True")

cmd_parts.append(f"--jinja_chat_template_override={TEMPLATE_REPO}")

if QUANTIZE:
    cmd_parts.append(f"--quantization_recipe={QUANTIZE}")

cmd_parts.append(f"-p {PREFILL_SEQ_LEN}")
cmd_parts.append(f"--cache_length {CACHE_LENGTH}")

cmd = " \\\n  ".join(cmd_parts)

print("📤 Çalıştırılacak komut:")
print("-" * 60)
print(cmd)
print("-" * 60)
print(f"\n🚀 Export başlıyor... (~30 dk)")

start_time = time.time()
env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN

process = subprocess.Popen(
    cmd.replace(" \\\n  ", " "),
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env
)

output_lines = []
for line in process.stdout:
    line = line.rstrip()
    output_lines.append(line)
    if any(kw in line.lower() for kw in
           ['error', 'warning', 'export', 'convert', 'quant', 'saving',
            'loading', 'done', 'complet', 'write', 'model', '%', 'step',
            'embedder', 'template', 'jinja', 'lm_head']):
        print(line)

process.wait()
elapsed = time.time() - start_time

print("\n" + "=" * 60)
if process.returncode == 0:
    print(f"✅ EXPORT BAŞARILI — {elapsed/60:.1f} dk sürdü")
    print(f"\n📁 {OUTPUT_DIR} içeriği:")
    for root, dirs, files in os.walk(OUTPUT_DIR):
        level = root.replace(OUTPUT_DIR, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath) / 1e6
            print(f"{'  ' * (level+1)}{f}  ({size:.1f} MB)")
else:
    print(f"❌ EXPORT BAŞARISIZ — Return code: {process.returncode}")
    print("\nSon 40 satır çıktı:")
    for line in output_lines[-40:]:
        print(line)
print("=" * 60)

## 4. Text inference testi

In [ ]:
import subprocess

LITERTLM_PATH = f"{OUTPUT_DIR}/model.litertlm"

# Fine-tune'un domain'ine uygun prompt — blind-assist için
TEXT_PROMPT = "Karşımda bir trafik ışığı var. Ne yapmalıyım?"

result = subprocess.run(
    f'litert-lm run "{LITERTLM_PATH}" --prompt="{TEXT_PROMPT}"',
    shell=True, capture_output=True, text=True, timeout=180
)

print("=" * 60)
print("TEST — Text only (fine-tune domain prompt)")
print("=" * 60)
print(f"Prompt: {TEXT_PROMPT}\n")

if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    print(f"❌ Hata (kod {result.returncode}):")
    print(result.stderr[-1000:])
else:
    pad_count = result.stdout.count('<pad>')
    if pad_count > 3:
        print(f"⚠️  <pad> sorunu ({pad_count} adet) — chat template kontrol et")
    else:
        print("✅ Text inference başarılı!")

## 5. Ara yükleme — Sade fine-tune modeli HF'ye yükle

Bu adım **opsiyonel** — sadece text-only modeli ayrı saklamak istersen. Asıl multimodal merge bir sonraki adımda.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=TARGET_REPO, repo_type="model", exist_ok=True)

print(f"🚀 Sade (text-only) model yükleniyor: {TARGET_REPO}")

api.upload_file(
    path_or_fileobj=LITERTLM_PATH,
    path_in_repo="model_text_only.litertlm",  # ← bu adla yüklenir, multimodal versiyon ayrı dosya
    repo_id=TARGET_REPO,
    repo_type="model",
    commit_message="Fine-tuned blind-assist Gemma 4 E2B (text-only export)",
)

print(f"✅ Yüklendi: https://huggingface.co/{TARGET_REPO}")

## 6. Vision merge — Resmi modelin vision encoder'ını ekle

Fine-tune modelimiz text-only export edildi. Multimodal yapmak için resmi modelin (`litert-community/gemma-4-E2B-it-litert-lm`) **vision encoder + adapter + end_of_vision** sections'ları + **LlmMetadataProto** + **SP_Tokenizer** alınıp birleştiriliyor.

**Not:** Resmi modelin SP_Tokenizer'ı kullanılır çünkü vision attention pattern'leri bu tokenizer'a göre kalibre edilmiş.

In [ ]:
import struct, flatbuffers, os
from huggingface_hub import hf_hub_download

MY_PATH       = LITERTLM_PATH
OFFICIAL_PATH = hf_hub_download(
    repo_id="litert-community/gemma-4-E2B-it-litert-lm",
    filename="gemma-4-E2B-it.litertlm"
)
OUTPUT_PATH   = "/tmp/blind_assist_merged.litertlm"
BLOCK_SIZE    = 16 * 1024

SECTION_TYPES = {
    0:"NONE", 1:"GenericBinaryData", 2:"Deprecated",
    3:"TFLiteModel", 4:"SP_Tokenizer", 5:"LlmMetadataProto",
    6:"HF_Tokenizer_Zlib", 7:"TFLiteWeights",
}

# ── PARSER ───────────────────────────────────────────────────────────
def read_string(buf, ptr_pos):
    rel = struct.unpack_from('<i', buf, ptr_pos)[0]
    sp = ptr_pos + rel
    ln = struct.unpack_from('<I', buf, sp)[0]
    return buf[sp+4:sp+4+ln].decode('utf-8', 'replace')

def read_kv(buf, kv_pos):
    vt_off = struct.unpack_from('<i', buf, kv_pos)[0]
    vt_pos = kv_pos - vt_off
    vt_size = struct.unpack_from('<H', buf, vt_pos)[0]
    num_fields = (vt_size - 4) // 2
    def get_field(fi):
        if fi >= num_fields: return None
        off = struct.unpack_from('<H', buf, vt_pos + 4 + fi*2)[0]
        return (kv_pos + off) if off else None
    key_p = get_field(0)
    key = read_string(buf, key_p) if key_p else ''
    val_table_ptr = get_field(2)
    val = ''
    if val_table_ptr:
        vtp = val_table_ptr + struct.unpack_from('<i', buf, val_table_ptr)[0]
        vt2_pos = vtp - struct.unpack_from('<i', buf, vtp)[0]
        str_off = struct.unpack_from('<H', buf, vt2_pos + 4)[0]
        if str_off:
            val = read_string(buf, vtp + str_off)
    return key, val

def parse_litertlm(path):
    with open(path, 'rb') as f:
        f.read(8); f.read(12); f.read(4)
        heo = struct.unpack('<Q', f.read(8))[0]
        fb_data = f.read(heo - f.tell())
    buf = bytearray(fb_data)
    def rtf(tp, fi):
        vt_pos = tp - struct.unpack_from('<i', buf, tp)[0]
        vt_size = struct.unpack_from('<H', buf, vt_pos)[0]
        fs = 4 + fi * 2
        if fs + 2 > vt_size: return None
        fo = struct.unpack_from('<H', buf, vt_pos + fs)[0]
        return (tp + fo) if fo else None
    root_pos = struct.unpack_from('<I', buf, 0)[0]
    sm_ptr = rtf(root_pos, 1)
    sm_pos = sm_ptr + struct.unpack_from('<i', buf, sm_ptr)[0]
    obj_ptr = rtf(sm_pos, 0)
    vec_pos = obj_ptr + struct.unpack_from('<i', buf, obj_ptr)[0]
    n = struct.unpack_from('<I', buf, vec_pos)[0]
    results = []
    for i in range(n):
        ep = vec_pos + 4 + i * 4
        op = ep + struct.unpack_from('<i', buf, ep)[0]
        p = rtf(op, 1); begin = struct.unpack_from('<Q', buf, p)[0] if p else 0
        p = rtf(op, 2); end   = struct.unpack_from('<Q', buf, p)[0] if p else 0
        p = rtf(op, 3); dt    = struct.unpack_from('<B', buf, p)[0] if p else 0
        kvs = []
        ip = rtf(op, 0)
        if ip:
            iv = ip + struct.unpack_from('<i', buf, ip)[0]
            ni = struct.unpack_from('<I', buf, iv)[0]
            for j in range(ni):
                kp = iv + 4 + j*4
                kpos = kp + struct.unpack_from('<i', buf, kp)[0]
                kvs.append(read_kv(buf, kpos))
        results.append({'idx':i,'data_type':dt,'begin':begin,'end':end,'labels':kvs})
    return results

def read_blob(path, begin, end):
    with open(path, 'rb') as f:
        f.seek(begin)
        return f.read(end - begin)

# ── FLATBUFFER BUILDER ────────────────────────────────────────────────
def build_flatbuffer_header(section_infos):
    b = flatbuffers.Builder(65536)
    def make_kv(key, val):
        val_str = b.CreateString(val)
        b.StartObject(1)
        b.PrependUOffsetTRelativeSlot(0, val_str, 0)
        val_table = b.EndObject()
        key_str = b.CreateString(key)
        b.StartObject(3)
        b.PrependUOffsetTRelativeSlot(0, key_str, 0)
        b.PrependByteSlot(1, 9, 0)
        b.PrependUOffsetTRelativeSlot(2, val_table, 0)
        return b.EndObject()

    section_offsets = []
    for info in section_infos:
        kv_offsets = [make_kv(k, v) for k, v in reversed(info['labels'])]
        if kv_offsets:
            b.StartVector(4, len(kv_offsets), 4)
            for kv in kv_offsets: b.PrependUOffsetTRelative(kv)
            items_vec = b.EndVector()
        b.StartObject(4)
        if kv_offsets: b.PrependUOffsetTRelativeSlot(0, items_vec, 0)
        b.PrependUint64Slot(1, info['begin'], 0)
        b.PrependUint64Slot(2, info['end'],   0)
        b.PrependByteSlot(3, info['data_type'], 0)
        section_offsets.append(b.EndObject())

    b.StartVector(4, len(section_offsets), 4)
    for off in reversed(section_offsets):
        b.PrependUOffsetTRelative(off)
    objects_vec = b.EndVector()

    b.StartObject(1)
    b.PrependUOffsetTRelativeSlot(0, objects_vec, 0)
    section_meta = b.EndObject()
    b.StartObject(2)
    b.PrependUOffsetTRelativeSlot(1, section_meta, 0)
    b.Finish(b.EndObject())
    return bytes(b.Output())

def align_to(offset, block=BLOCK_SIZE):
    return (offset + block - 1) // block * block

def compute_layout(fb_size, blobs):
    cursor = align_to(32 + fb_size)
    layout = []
    for blob in blobs:
        begin = cursor; end = begin + len(blob)
        layout.append((begin, end))
        cursor = align_to(end)
    return layout

# ── PARSE ─────────────────────────────────────────────────────────────
print("📖 Modeller parse ediliyor...")
my_sections  = parse_litertlm(MY_PATH)
off_sections = parse_litertlm(OFFICIAL_PATH)

print("\nKendi fine-tune modelim:")
for s in my_sections:
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {(s['end']-s['begin'])/1e6:>8.1f} MB  {s['labels']}")

print("\nResmi model:")
for s in off_sections:
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {(s['end']-s['begin'])/1e6:>8.1f} MB  {s['labels']}")

# ── BLOB'LARI OKU ────────────────────────────────────────────────────
print("\n📦 Blob'lar okunuyor...")
my_blobs = []
for i, s in enumerate(my_sections):
    if s['data_type'] == 5:  # LlmMetadataProto → resmi modelden al (vision config içerir)
        blob = read_blob(OFFICIAL_PATH, off_sections[0]['begin'], off_sections[0]['end'])
        print(f"  [{i}] LlmMetadataProto → resmi modelden ({len(blob)} bytes)")
    else:
        blob = read_blob(MY_PATH, s['begin'], s['end'])
        print(f"  [{i}] {SECTION_TYPES.get(s['data_type'],'?')} → fine-tune modelden ({len(blob)/1e6:.1f} MB)")
    my_blobs.append(blob)

# Resmi modelden vision sections (encoder, adapter, end_of_vision)
vision_indices = [7, 8, 9]
vision_blobs   = [read_blob(OFFICIAL_PATH, off_sections[i]['begin'], off_sections[i]['end']) for i in vision_indices]
vision_labels  = [off_sections[i]['labels'] for i in vision_indices]
vision_dtypes  = [off_sections[i]['data_type'] for i in vision_indices]

print("\n📦 Vision blob'ları (resmi modelden):")
for i, vi in enumerate(vision_indices):
    print(f"  Resmi [{vi}] → {len(vision_blobs[i])/1e6:.1f} MB  {vision_labels[i]}")

# ── BİRLEŞTİR ────────────────────────────────────────────────────────
all_blobs  = my_blobs + vision_blobs
all_dtypes = [s['data_type'] for s in my_sections] + vision_dtypes
all_labels = [s['labels']    for s in my_sections] + vision_labels

# İki geçişli layout (FlatBuffer header boyutu blob offset'lerine bağlı)
dummy_infos = [{'begin':0,'end':0,'data_type':dt,'labels':lb} for dt,lb in zip(all_dtypes, all_labels)]
dummy_fb    = build_flatbuffer_header(dummy_infos)
layout      = compute_layout(len(dummy_fb), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)
layout2     = compute_layout(len(fb_bytes), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout2,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)

print(f"\n✍️  Yazılıyor → {OUTPUT_PATH}")
with open(OUTPUT_PATH, 'wb') as out:
    out.write(b'LITERTLM')
    out.write(struct.pack('<III', 1, 5, 0))
    out.write(b'\x00' * 4)
    out.write(struct.pack('<Q', 32 + len(fb_bytes)))
    out.write(fb_bytes)
    for blob, (begin, end) in zip(all_blobs, layout2):
        cur = out.tell()
        if cur < begin: out.write(b'\x00' * (begin - cur))
        out.write(blob)

print(f"✅ Tamamlandı: {os.path.getsize(OUTPUT_PATH)/1e9:.2f} GB")

# ── DOĞRULA ──────────────────────────────────────────────────────────
print("\n📋 Doğrulama:")
merged_sections = parse_litertlm(OUTPUT_PATH)
for s in merged_sections:
    size = (s['end'] - s['begin']) / 1e6
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {size:>8.1f} MB  {s['labels']}")

## 7. Doğrulama testleri

In [ ]:
# LlmMetadataProto'nun resmi modelden geldiğini doğrula
merged = parse_litertlm(OUTPUT_PATH)
s = merged[0]
actual_size = s['end'] - s['begin']
official_meta_size = off_sections[0]['end'] - off_sections[0]['begin']
print(f"Section [0] LlmMetadataProto boyut: {actual_size} bytes")
print(f"Resmi modeldeki boyut: {official_meta_size} bytes")
print("✅ OK" if actual_size == official_meta_size else f"❌ HATA — eşleşmiyor")

# Vision encoder bytes karşılaştır
with open(OFFICIAL_PATH, 'rb') as f:
    f.seek(off_sections[7]['begin'])
    off_vision = f.read(64)

vision_idx_in_merged = next(i for i, s in enumerate(merged)
                            if any(k == 'model_type' and v == 'tf_lite_vision_encoder'
                                   for k, v in s['labels']))
with open(OUTPUT_PATH, 'rb') as f:
    f.seek(merged[vision_idx_in_merged]['begin'])
    our_vision = f.read(64)

print(f"\nVision encoder ilk 64 byte eşleşiyor mu: {off_vision == our_vision}")
print("✅ Vision encoder doğru kopyalandı" if off_vision == our_vision else "❌ HATA")

## 8. Final upload — Multimodal modeli HF'ye yükle

Bu Gallery'nin allowlist'inde `modelFile="model.litertlm"` adıyla beklediği dosya.

## 9. Akıllı merge — Resmi modelin gövdesini kullan, sadece prefill_decode'u fine-tune ile değiştir

Önceki merge (fine-tune embedder + base vision) embedding uzayı uyumsuzluğu nedeniyle SEGV veriyordu. Bu sefer **tersini** yapıyoruz:

- **Resmi modelden:** LlmMetadataProto, SP_Tokenizer, embedder, per_layer_embedder, vision_encoder, vision_adapter, end_of_vision
- **Fine-tune'dan:** SADECE `tf_lite_prefill_decode` (transformer body)

Böylece tüm embedding uzayı tutarlı kalır, sadece transformer ağırlıkları senin fine-tune'unun. Audio ve mtp_drafter bırakıldı (yer kazandırır + base versiyonu fine-tune ile uyumsuz olabilir).

In [ ]:
import struct, os
# parse_litertlm, build_flatbuffer_header, compute_layout, read_blob, BLOCK_SIZE,
# SECTION_TYPES, MY_PATH, OFFICIAL_PATH zaten önceki cell'lerden tanımlı.

OUTPUT_PATH_V2 = "/tmp/blind_assist_smart_merge.litertlm"

# Resmi model section'larını parse et
off = parse_litertlm(OFFICIAL_PATH)
my  = parse_litertlm(MY_PATH)

print("Resmi model section'ları:")
for s in off:
    label = s['labels'][0][1] if s['labels'] else ''
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {(s['end']-s['begin'])/1e6:>8.1f} MB  {label}")

print("\nFine-tune section'ları:")
for s in my:
    label = s['labels'][0][1] if s['labels'] else ''
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {(s['end']-s['begin'])/1e6:>8.1f} MB  {label}")

# Fine-tune'da prefill_decode'u bul
ft_prefill_idx = next(i for i, s in enumerate(my)
                      if any(k == 'model_type' and v == 'tf_lite_prefill_decode'
                             for k, v in s['labels']))
print(f"\nFine-tune prefill_decode bulundu: section [{ft_prefill_idx}] "
      f"({(my[ft_prefill_idx]['end']-my[ft_prefill_idx]['begin'])/1e6:.1f} MB)")

# Hedef section sırası (runtime'ın beklediği)
# Resmi modelden alınacak: 0=LlmMeta, 1=SP_Tok, 2=embedder, 3=per_layer_emb,
#                          7=vision_enc, 8=vision_adapter, 9=end_of_vision
official_indices = [0, 1, 2, 3, 7, 8, 9]
official_blob_info = [(off[i]['data_type'], off[i]['labels'],
                       read_blob(OFFICIAL_PATH, off[i]['begin'], off[i]['end']))
                      for i in official_indices]

# Fine-tune'dan: prefill_decode
ft_prefill_blob = read_blob(MY_PATH, my[ft_prefill_idx]['begin'], my[ft_prefill_idx]['end'])
ft_prefill_info = (my[ft_prefill_idx]['data_type'],
                   my[ft_prefill_idx]['labels'],
                   ft_prefill_blob)

# Final sıra: resmi[0..3] + vision[7..9] + fine-tune prefill_decode
all_info = official_blob_info + [ft_prefill_info]

print("\n📋 Birleştirilecek sırlama:")
for i, (dt, lb, blob) in enumerate(all_info):
    label = lb[0][1] if lb else SECTION_TYPES.get(dt, '?')
    src = "FINE-TUNE" if i == len(all_info) - 1 else "official"
    print(f"  [{i}] {label:<35} {len(blob)/1e6:>8.1f} MB  ({src})")

# Layout hesapla
all_blobs  = [info[2] for info in all_info]
all_dtypes = [info[0] for info in all_info]
all_labels = [info[1] for info in all_info]

dummy_infos = [{'begin':0,'end':0,'data_type':dt,'labels':lb} for dt,lb in zip(all_dtypes, all_labels)]
dummy_fb    = build_flatbuffer_header(dummy_infos)
layout      = compute_layout(len(dummy_fb), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)
layout2     = compute_layout(len(fb_bytes), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout2,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)

print(f"\n✍️  Yazılıyor → {OUTPUT_PATH_V2}")
with open(OUTPUT_PATH_V2, 'wb') as out:
    out.write(b'LITERTLM')
    out.write(struct.pack('<III', 1, 5, 0))
    out.write(b'\x00' * 4)
    out.write(struct.pack('<Q', 32 + len(fb_bytes)))
    out.write(fb_bytes)
    for blob, (begin, end) in zip(all_blobs, layout2):
        cur = out.tell()
        if cur < begin: out.write(b'\x00' * (begin - cur))
        out.write(blob)

print(f"✅ Tamamlandı: {os.path.getsize(OUTPUT_PATH_V2)/1e9:.2f} GB")

# Doğrulama
print("\n📋 Yeni model section'ları:")
for s in parse_litertlm(OUTPUT_PATH_V2):
    label = s['labels'][0][1] if s['labels'] else ''
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {(s['end']-s['begin'])/1e6:>8.1f} MB  {label}")

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()

# Repo yoksa oluştur (Bölüm 5 atlandıysa burada yaratılır)
api.create_repo(repo_id=TARGET_REPO, repo_type="model", exist_ok=True)
print(f"📦 Repo hazır: {TARGET_REPO}")

print(f"🚀 Multimodal merged model yükleniyor: {TARGET_REPO}")

api.upload_file(
    path_or_fileobj=OUTPUT_PATH,
    path_in_repo="model.litertlm",
    repo_id=TARGET_REPO,
    repo_type="model",
    commit_message="Blind-assist fine-tune + official vision encoder (multimodal)",
)

size_gb = os.path.getsize(OUTPUT_PATH) / 1e9
print(f"\n✅ Yüklendi: https://huggingface.co/{TARGET_REPO}")
print(f"   Dosya: model.litertlm ({size_gb:.2f} GB)")
print(f"\n📱 Şimdi Gallery'nin ModelManagerViewModel.kt allowlist'ini güncelle:")
print(f'      "modelId": "{TARGET_REPO}",')
print(f'      "modelFile": "model.litertlm",')
print(f'      "sizeInBytes": {os.path.getsize(OUTPUT_PATH)},')